# Setting up Controller and Event Analyst

Hello! This tutorial will show you how to set up `Ralph` to perform the tasks for you!

`Ralph` can:
* pre-process the light curves to make sure the fitting process runs smoothly;
* fit ongoing and finished gravitational microlensing events;
* create a color-magnitude diagram for all found best-fitting models.

`Ralph` is pretty flexible, and you can and may turn on and off certain modules.

## Overview

Ralph has two basic classes: a Controller and an Analyst.

The **Controller** manages the workflow for all events you'd like to analyze. It controls how many events are handled at once,
where the inputs and output files go, and how, where, at what level the logs are created. For each event you'd like to analyze,
the Controller will launch an **Event Analyst**.

The **Event Analyst** controls the analysis workflow for a single event. It handles sub-analysts (Light Curve, Fit, and CMD). Event Analyst launches respective sub-analysts based on the Event Analyst configuration for each event. It knows where to store the outputs and the logs.
For each event, you may or may not configure your Event Analyst differently. If you have a set of events, each with a varying data quality, you can set up their pre-processing or fitting procedures differently.
Or, if you have color-magnitude diagram information for only some of the events, or only some of the surveys, you can set up too!
Each event can be treated individually!

The **Event Analyst** manages three types of Analysts:
* the **Light Curve Analyst** -- it's handling the pre-processing of the light curves, to make sure no invalid entries prevent
running the fitting process smoothly;
* the **Fit Analyst** -- it's handling the fitting procedure;
* the **CMD Analyst** -- it's handling the creation of the color-magnitude diagram for each event and catalog you have specified.


## Configuring the Controller

Okay! Let's start with something simple and set up our controller to run on a single event.
First, we have to let `Ralph` know how to set up the Controller.
You can pass a YAML or JSON file with the configuration, but you can also pass it as a dictionary.
We will create a dictionary to make it easier in this tutorial.

In [1]:
import os

# Here we set up the path to the tutorial folder, to ensure that the example executes correctly
current_path = os.getcwd()
os.chdir("../../")
ralph_home_dir = os.getcwd()

In [2]:
ralph_home_dir

'/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph'

In this tutorial, we have already set up the necessary directories. For your own project, you will have to do them up for yourself. `Ralph` requires a specific directory and file structure to work:

```
events_path/
│
├── event_1/
│   └── config.json
├── event_2/
│   └── config.json
├── event_3/
│   └── config.json
...
```
In the case of this tutorial, the file structure looks like this:
```
ralph_home_directory/examples/controller/
│
├── OGLE_2016_BLG_1395/
│   └── config.json (we will create this file in a moment)
```

The `config` for the Controller requires the following keywords:
* `python_compiler` - name of the Python compiler you want to use for this run; it can be just "python" to use a default compiler, but you can also specify python3, python3.10, python3.12, etc. or a path to your Virtual Environment's Python compiler;
* `group_processing_limit` - group processing limit refers to how many events will be analyzed at the same time; it is limited by how many cores are available on your machine;
* `config_type` - what file format (YAML or JSON) will you use for your configuration files for the Event Analysts;
* `events_path` - where will the outputs be saved, and where Event Analysts should look for the configuration files for each event; you can pass it as a string, or use the power of the `os` package when creating a dictionary;
* `software_dir` -  because `Ralph` uses the subprocess package, it requires a specific setup; you have to specify what is the path to your Ralph's repository location of the Analysts' source files;
* `log_stream` - `Ralph` uses `logging.Logger` instance to handle logging statements for the Controller; if `log_stream` is set to `True`, it can be streamed to the standard output, and then captured with, for example, Kubernetes; if set to `False`, the log will be saved to the location specified in log_location;
* `log_location` - where you'd like to save the file with controller logs; the file will appear in the log_location directory, under the name `controller.log`;
* `log_level` - how verbose should the log statements be; there are three levels available: `error` (least verbose), `info` (medium), `debug` (very verbose).

In [15]:
config = {
    "python_compiler": "python",
    "group_processing_limit": 2,
    "config_type": "json",
    "events_path": os.path.join(ralph_home_dir, "examples", "controller"),
    "software_dir": os.path.join(ralph_home_dir, "src", "ralph", "analyst"),
    "log_stream": True,
    "log_location": os.path.join(ralph_home_dir, "examples", "controller"),
    "log_level": "info",
}

On top of the configuration information, `Ralph` also needs to know which events you'd like to have analyzed.
For the sake of simplicity, we will analyze only one event.

These event names will correspond to folder names inside the `events_path` location.

<!-- * If you pass the Event Analyst configuration as dictionaries created on the fly,
`Ralph` will create folders with event names inside the `events_path` directory. -->
* If you pass the Event Analyst configuration as files, they have to be located inside the
`events_path` folder, in a folder with the same name as your event name, under `config.yaml`
or `config.json` (whichever file format you specified in the Controller configuration under
`config_type`).

In [16]:
# We will analyze only one event from the OGLE survey
event_list = [
    "OGLE_2016_BLG_1395",
]

Great! Now we have to set up our Event Analyst configuration and input files, before we can move forward.

## Configuring the Event Analyst

Let's say you only want to perform some basic fitting with `Ralph` for your event. 
We will show you the minimum information necessary to launch a fitting process for an event.

This configuration file can, but doesn't have to include the following keywords:
* `lc_analyst` - it should hold a dictionary with the Light Curve Analyst set up.
If you leave this as an empty dictionary, it will launch the Light Curve Analyst with the default setup.
You can also drop this keyword completely, to not run the Light Curve Analyst at all, but it's not recommended
if you want to run the Fit Analyst.
* `fit_analyst` - it should hold a dictionary with the Fit Analyst set up. You can leave it empty, and it will
use the default setup for fitting. The default is `pyLIMA` as a fitting package, and
the [**Trust Region Reflective**](https://github.com/ebachelet/pyLIMA/blob/master/pyLIMA/fits/TRF_fit.py) fitting
routine for all types of models, except the `PSPL_no_blend_no_piE` (point source-point lens model without blending or parallax effect), where the [**Differential Evolution**](https://github.com/ebachelet/pyLIMA/blob/master/pyLIMA/fits/DE_fit.py) is used.
You can also drop this keyword completely, to not run the Fit Analyst at all,
* `cmd_analyst` - it should hold a dictionary with the CMD Analyst set up. Unlike the two previous Analysts, it doesn't have a default setup.
If you'd like to run the CMD Analyst for an event, you **have** to provide the configuration.

The least you should specify is the `fit_analyst` configuration, otherwise `Ralph` cannot help you model microlensing events.

### The bare minimum

In this example, we will run the Light Curve Analyst with the default setup, and we will customize the Fit Analyst (in a moment).
The `event_analyst_config` accepts the following keywords:
* `event_name` - event name, should match an event name provided in `event_list` (mandatory field);
* `ra` - event Right Ascension, in degrees, necessary for the microlensing parallax model (mandatory field);
* `dec` - event Declination, in degrees,necessary for the microlensing parallax model (mandatory field);
* `lc_analyst` - a dictionary with the Light Curve Analyst setup; we will leave it empty, to run with the default setup (optional field);
* `fit_analyst` - a dictionary with the Fit Analyst setup; we will leave it empty for now, because it's the most complex one (optional field);
* `light_curves` - a list of dictionaries with light curves (mandatory field); it has to contain a list of dictionaries with these keywords:
    * `survey` - survey or telescope name, which was used to obtain the data (mandatory field);
    * `band` - band (or filer) in which the data was obtained (mandatory field);
    * `path` - a path to a location where the light curve is stored (mandatory field);
    * `ephemeris` - a path to a location where the ephemeris for the observatory is located (optional field).

In [17]:
event_analyst_config = {
    "event_name": "OGLE_2016_BLG_1395",
    "ra": 271.1194,
    "dec": -29.8162,
    "lc_analyst": {}, # We will leave it empty, to run with the default setup.
    "fit_analyst": {},
    "light_curves": [
        {
            "survey": "OGLE",
            "band": "I",
            "path": os.path.join(
                ralph_home_dir, "examples", "example_light_curve_OB161395_I.dat"
            )
        },
    ]
}

The file with the light curve has to contain three columns:
* first column with Julian Day of each taken data point;
* second column with magnitude of each taken data point;
* third column with uncertainty of magnitude for each taken data point;
Let's have a look at the example:

In [18]:
import numpy as np

data = np.loadtxt(os.path.join(ralph_home_dir, "examples", "example_light_curve_OB161395_I.dat"))
data

array([[2.45526088e+06, 1.49340000e+01, 5.05800000e-03],
       [2.45526181e+06, 1.49360000e+01, 5.06200000e-03],
       [2.45526285e+06, 1.49430000e+01, 8.50800000e-03],
       ...,
       [2.45892387e+06, 1.49270000e+01, 5.04300000e-03],
       [2.45892389e+06, 1.49330000e+01, 5.05600000e-03],
       [2.45892389e+06, 1.49320000e+01, 5.05400000e-03]], shape=(2780, 3))

### Fit Analyst configuration

Now let's focus on the configuration of the Fit Analyst. It can get quite complex, so we will go step by step.

Fit Analyst has four major keywords:
* `ongoing_magnification_threshold` (default value: 1.05) -- threshold for magnification, if the magnification of the last data point for a simple point source-point lens model (PSPL) is larger, the event is considered as ongoing;
* `ongoing_amplitude_threshold` (default value: 1.0 mag) -- threshold for amplitude from baseline of the last data point; if the amplitude is larger, the event is considered as ongoing;
* `time_of_peak_bin_size` (default value: 2.0 days) --  width in days used to calculate the median magnitude of points within each bin; the binned light curve is used to estimate the approximate time of peak of the event;
* `model_fit_configuration` -- a dictionary with fit configuration, we will come back to it.

In [19]:
event_analyst_config["fit_analyst"] = {
    # Threshold for PSPL model magnification
    "ongoing_magnification_threshold" : 1.10,
    # Threshold for amplitude
    "ongoing_amplitude_threshold": 1.0,
    # Bin size when looking for the time of peak
    "time_of_peak_bin_size": 1.0,
    # Configuration of each fit for specific models
    "model_fit_configuration" : {}
}

You can configure the Fit Analyst for the following supported model types:

* `PSPL_no_blend_no_piE` -- point source point lens model without blending and microlensing parallax effect;
* `PSPL_blend_no_piE` -- point source point lens model with blending and without microlensing parallax effect;
* `PSPL_blend_piE` -- point source point lens model with blending and microlensing parallax effect;
* `PSPL_no_blend_piE` -- point source point lens model without blending and with microlensing parallax effect;

For each model type, you can provide a dictionary with the following keywords:
* `fitting_package` (default value: pyLIMA) -- the name of the fitting package you want to use for this model; ⚠️currently, only pyLIMA is supported⚠️;
* `fitting_method` (default value: TRF) -- the name of the fitting method supported by the `fitting_package` for this model, available options: TRF (Trust Reflective Function), DE (Differential Evolution);
* `boundaries` -- a dictionary with a list of lower and upper boundaries for each microlensing model parameter (`t0`, `u0`, `tE`, `piEN`, `piEE`) you'd like to modify.

Below, we will set the fitting package for each model, and then specify more parameters for the `PSPL_blend_piE` model.

In [20]:
model_list = [
    "PSPL_no_blend_no_piE",
    "PSPL_blend_no_piE",
    "PSPL_blend_piE",
    "PSPL_no_blend_piE",
]

for model in model_list:
    if model != "PSPL_blend_piE":
        event_analyst_config["fit_analyst"]["model_fit_configuration"][model] =  {
            "fitting_package": "pyLIMA",
        }
    else:
        event_analyst_config["fit_analyst"]["model_fit_configuration"][model] =  {
            "fitting_package": "pyLIMA",
            "fitting_method": "TRF", 
            "boundaries": {
                "u0": [ -2.0, 2.0 ],
                "piEN": [ -1.0, 1.0 ],
                "piEE": [ -1.0, 1.0 ],
            }
        }

Controller looks for Event Analyst configuration in a very specific way. It checks the directory specified in the Controller `config["events_path"]`. There it looks for subdirectories named like the events passed in the `event_list` variable of the Controller. Such subdirectory has to contain a file called `config.json` with the Event Analyst configuration.

To pass our Event Analyst configuration dictionary to Controller, we will have to turn it into a JSON, and then save it in a location where the Controller will look for files.

In [21]:
import json

config_path = os.path.join(
    ralph_home_dir, "examples", "controller", "OGLE_2016_BLG_1395", "config.json"
)
with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(event_analyst_config, f, ensure_ascii=False, indent=4)

## Initializing the Controller

Now we can initialize the Controller.

In [22]:
from ralph.controller.controller import Controller

controller = Controller(event_list, config_dict=config)

2026-06-18 18:09:33.533 - INFO - Processing started. Opened log.
2026-06-18 18:09:33.533 - INFO - Processing started. Opened log.


Let's look up the Controller configuration:

In [23]:
for key in controller.config:
    print(key, ":", controller.config[key])

python_compiler : python
group_processing_limit : 2
config_type : json
events_path : /home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph/examples/controller
software_dir : /home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph/src/ralph/analyst
log_stream : True
log_location : /home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph/examples/controller
log_level : info


Now, we can launch the Controller, and see what it's doing through the logs.

In [24]:
controller.launch_analysts()

2026-06-18 18:09:34.957 - INFO - Controller: Start processing.
2026-06-18 18:09:34.957 - INFO - Controller: Start processing.
2026-06-18 18:09:34.960 - INFO - Controller: Commands created. Spawning processes.
2026-06-18 18:09:34.960 - INFO - Controller: Commands created. Spawning processes.
2026-06-18 18:09:34.988 - INFO - About to start subprocess for event: 
2026-06-18 18:09:34.988 - INFO - About to start subprocess for event: 
2026-06-18 18:09:34.992 - INFO - Command: ['python', '/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph/src/ralph/analyst/event_analyst.py', '--event_name', 'OGLE_2016_BLG_1395', '--analyst_path', '/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph/examples/controller/OGLE_2016_BLG_1395/', '--log_level', 'info', '--stream', 'True']
2026-06-18 18:09:34.992 - INFO - Command: ['python', '/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/ralph/src/ralph/analyst/event_analyst.py', '--event_name', 'OGLE_2016_BLG_1395', 

/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = ss.anderson(sample)


2026-06-18 18:09:38.196 - INFO - Fit Analyst: Plotting finished.
2026-06-18 18:09:38.229 - INFO - Fit Analyst:  Finished fitting.
2026-06-18 18:09:38.229 - INFO - Fit Analyst: Identify ongoing event.
2026-06-18 18:09:38.229 - INFO - Fit Analyst: Performing a finished event fit.
2026-06-18 18:09:38.229 - INFO - Fit Analyst: Starting finished event fit.
2026-06-18 18:09:38.229 - INFO - Fit Analyst: Finding PSPL starting parameters.
2026-06-18 18:09:38.229 - INFO - Fit Analyst:Performing PSPL with blend fit.
2026-06-18 18:09:38.229 - INFO - Fit Analyst: Using fitting setup specified by the User.
check_event  : Everything looks fine...
2026-06-18 18:09:38.260 - INFO - Fit Analyst -- pyLIMA: Fitting without microlensing parallax.
2026-06-18 18:09:38.260 - INFO - Fit Analyst -- pyLIMA: Using default fitting method (TRF).
2026-06-18 18:09:38.294 - INFO - Fit Analyst -- pyLIMA: Using boundaries default for ralph.
2026-06-18 18:09:38.295 - INFO - Fit Analyst -- pyLIMA: Adding starting parameter

/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = ss.anderson(sample)


2026-06-18 18:09:39.770 - INFO - Fit Analyst: Plotting finished.
2026-06-18 18:09:39.813 - INFO - Fit Analyst: Finished fitting PSPL with blend fit.
2026-06-18 18:09:39.813 - INFO - Fit Analyst: Performing PSPL+piE fit.
2026-06-18 18:09:39.813 - INFO - Fit Analyst: Starting fitting model PSPL_blend_piE_p
2026-06-18 18:09:39.813 - INFO - Fit Analyst: Using fitting setup specified by the User.
check_event  : Everything looks fine...
2026-06-18 18:09:39.839 - INFO - Fit Analyst -- pyLIMA: Fitting with microlensing parallax.
Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-18 18:09:40.301 - INFO - Fit Analyst -- pyLIMA: Fitting method: TRF.
2026-06-18 18:09:40.340 - INFO - Fit Analyst -- pyLIMA: Using boundaries passed by the User.
2026-06-18 18:09:40.340 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters:
2026-06-18 18:09:40.340 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters: t0 = 2457721.453
2026-06-18 18:09:40.340 - INFO - Fit Analyst -- pyLIMA: Add

...

initial_guess  : Initial parameters guess SUCCESS
Using guess:  [2457721.453, 0.58404, 102.487, 0.0, 0.0, 42934.80463637442, 96733.55518228043]
Trust Region Reflective fit SUCCESS
best model:
OrderedDict([('t0', np.float64(2457711.4237656007)),
             ('u0', np.float64(0.16153958654689463)),
             ('tE', np.float64(362.22592865834775)),
             ('piEN', np.float64(-0.05499580230427389)),
             ('piEE', np.float64(-0.10405649814225379)),
             ('fsource_OGLE_I', np.float64(6138.116605664898)),
             ('ftotal_OGLE_I', np.float64(96708.16522707332)),
             ('soft_l1', np.float64(2028.778298105327))])
2026-06-18 18:09:40.564 - INFO - Fit Analyst -- pyLIMA: Fitting finished
2026-06-18 18:09:40.567 - INFO - Fit Analyst: Plots: grabbing colours and markers.
2026-06-18 18:09:40.568 - INFO - Fit Analyst: Starting a plot.


/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = ss.anderson(sample)


Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-18 18:09:42.634 - INFO - Fit Analyst: Plotting finished.
2026-06-18 18:09:42.695 - INFO - Fit Analyst:  Finished fitting model PSPL_blend_piE_p
2026-06-18 18:09:42.695 - INFO - Fit Analyst: Starting fitting model PSPL_blend_piE_n
2026-06-18 18:09:42.695 - INFO - Fit Analyst: Using fitting setup specified by the User.
check_event  : Everything looks fine...
2026-06-18 18:09:42.721 - INFO - Fit Analyst -- pyLIMA: Fitting with microlensing parallax.
Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-18 18:09:42.818 - INFO - Fit Analyst -- pyLIMA: Fitting method: TRF.
2026-06-18 18:09:42.855 - INFO - Fit Analyst -- pyLIMA: Using boundaries passed by the User.
2026-06-18 18:09:42.855 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters:
2026-06-18 18:09:42.855 - INFO - Fit Analyst -- pyLIMA: Adding starting parameters: t0 = 2457711.424
2026-06-18 18:09:42.855 - INFO - Fit Analyst -- pyLIMA: Adding start

/home/katarzyna/Documents/Microlensing_Fitting_Pipeline/ralph/.venv/lib/python3.12/site-packages/pyLIMA/fits/stats.py:48: FutureWarning: As of SciPy 1.17, users must choose a p-value calculation method by providing the `method` parameter. `method='interpolate'` interpolates the p-value from pre-calculated tables; `method` may also be an instance of `MonteCarloMethod` to approximate the p-value via Monte Carlo simulation. When `method` is specified, the result object will include a `pvalue` attribute and not attributes `critical_value`, `significance_level`, or `fit_result`. Beginning in 1.19.0, these other attributes will no longer be available, and a p-value will always be computed according to one of the available `method` options.
  AD_stat, AD_critical_values, AD_significance_levels = ss.anderson(sample)


Parallax(Full) estimated for the telescope OGLE_I: SUCCESS
2026-06-18 18:09:45.277 - INFO - Fit Analyst: Plotting finished.
2026-06-18 18:09:45.338 - INFO - Fit Analyst:  Finished fitting model PSPL_blend_piE_n
2026-06-18 18:09:45.339 - INFO - Event Analyst: Processing finished.
2026-06-18 18:09:45.339 - INFO - -------------------------------------------
2026-06-18 18:09:45.339 - INFO - Processing complete.

2026-06-18 18:09:45.959 - INFO - Controller: Processing finished.
2026-06-18 18:09:45.959 - INFO - Controller: Processing finished.
2026-06-18 18:09:45.962 - INFO - Processing complete.

2026-06-18 18:09:45.962 - INFO - Processing complete.

2026-06-18 18:09:45.964 - INFO - Processing complete.

2026-06-18 18:09:45.964 - INFO - Processing complete.



After running the Controller, you should see the following files in the folder with the event:

In [25]:
event_dir = os.path.join(ralph_home_dir, "examples", "controller", "OGLE_2016_BLG_1395")

for file in os.listdir(event_dir):
    print(file)

fit_results.json
PSPL_blend_no_piE.html
config.json
PSPL_blend_piE_n.html
.ipynb_checkpoints
fit_stats.txt
PSPL_no_blend_no_piE.html
PSPL_blend_piE_p.html


You can look up fitting results in `fit_results.json`, while `fit_stats.txt` contains useful statistics you can use to judge which model is the best.